# Scaling curve on Kaggle

Runs the training half of `isaac-grasp-scaling` on Kaggle's free GPU quota.

**Set the accelerator to `GPU T4 x2`** (Settings, right-hand panel). P100 also works
for training; it is only Isaac Sim that needs RT cores, and Isaac Sim is not used here.

## What this notebook does and does not do

| stage | where |
|---|---|
| Generate the dataset | **not here.** MuJoCo on 4 CPU cores gives about 60k samples/hour; Isaac Lab needs a rented RTX box. Generate elsewhere and upload the result as a Kaggle Dataset. |
| Train the network at each size | **here**, on the T4 |
| Evaluate, and re-run the heuristic control | **here**, but on CPU: the evaluator is MuJoCo physics, so it does not benefit from the GPU |

Evaluation is therefore the slow part on Kaggle, at roughly a second per episode
across 4 cores. Budget for it: 200 episodes per split, two splits per point.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())


## 2. Install

`libosmesa6` is not optional. Kaggle images have no display and no GL driver, so
MuJoCo needs a software renderer for the evaluation stage. Without it every script
dies at the first render with no useful message.


In [ ]:
!apt-get -qq update && apt-get -qq install -y libosmesa6 libgl1 > /dev/null
!pip -q install 'isaacgrasp @ git+https://github.com/abyyworld/isaac-grasp-scaling.git'

import os
# Software rendering. Set before anything imports mujoco.
os.environ['MUJOCO_GL'] = 'osmesa'


## 3. Fetch the robot meshes and verify by running

`check_setup.py` renders a frame, executes a grasp and runs the collector. If it
passes, the environment is genuinely working rather than merely installed.


In [ ]:
!git clone -q https://github.com/abyyworld/isaac-grasp-scaling.git /kaggle/working/repo
%cd /kaggle/working/repo
!python scripts/fetch_assets.py
!python scripts/check_setup.py


## 4. Point at the dataset

Upload the dataset you generated elsewhere as a Kaggle Dataset and attach it to this
notebook, then set `DATA` to its path. A 6,000-scene MuJoCo dataset at 224 px is
about 4.3 GB.

To generate a small one here instead, uncomment the second cell. It is CPU-bound and
slow: about 60k samples per hour on Kaggle's 4 cores.


In [ ]:
DATA = '/kaggle/input/YOUR-DATASET-SLUG/mj18k'   # <-- edit this

import json, pathlib
meta = json.loads((pathlib.Path(DATA) / 'dataset_meta.json').read_text())
print(meta['stats']['n'], 'samples from', meta['n_scenes'], 'scenes,',
      meta['angles_per_scene'], 'grasps per scene')


In [ ]:
# Optional: generate a small dataset here instead of uploading one.
# !python scripts/collect.py --backend mujoco --scenes 2000 --workers 4 \
#     --angles-per-scene 3 --split seen --out /kaggle/working/data/mj6k
# DATA = '/kaggle/working/data/mj6k'


## 5. Run the curve

This is what the GPU is for. The settings below are the ones the original study said
it needed and could not afford on a laptop CPU: full 224 px resolution, 30 epochs,
ImageNet initialisation.

Changing them from the committed MuJoCo arm's settings is deliberate and it means the
two arms are not directly comparable point for point. What stays comparable is the
shape of each curve, and the control, which is re-run inside each arm.

The run is resumable: every point writes `point.json` and is skipped on a re-run, so a
12-hour session timeout costs you one point rather than the whole curve. Commit the
notebook output and re-run to continue.


In [ ]:
!python scripts/run_scaling.py \
    --data {DATA} \
    --out /kaggle/working/results/kaggle \
    --sizes 512 1024 2048 4096 \
    --epochs 30 --input-size 224 --batch-size 32 --pretrained \
    --device cuda --workers 2 --eval-workers 4 --eval-episodes 200


## 6. Results

`scaling.json`, `scaling.csv` and `scaling_curve.png` land in `/kaggle/working/results/kaggle`.
Download them and commit them to the repository under `results/scaling/kaggle/`.


In [ ]:
from IPython.display import Image, display
import pandas as pd

OUT = '/kaggle/working/results/kaggle'
display(pd.read_csv(f'{OUT}/scaling.csv'))
display(Image(f'{OUT}/scaling_curve.png'))


In [ ]:
import json
reading = json.load(open(f'{OUT}/scaling.json'))['reading']
held = reading['trends']['held_out_success']
print(f"held-out success: {held['slope_per_doubling_pp']:+.2f} points per doubling"
      f" (r2 {held['r_squared']:.2f}, {held['n_points']} points)")
extrapolation = reading.get('extrapolation')
if extrapolation:
    print(extrapolation['note'])
    print(f"projected samples to reach the control: {extrapolation['projected_samples']:,.0f}")
